# nb_silver_santos_curso_motoristas
## Camada: Silver → Gold

**Fluxo deste notebook:**
1. Lê os Parquets silver brutos (gerados pelo nb_ingest)
2. Normaliza nomes de colunas (snake_case, sem acentos)
3. Consolida colunas duplicadas geradas pela API (bfill)
4. Converte tipos (datas, numéricos)
5. Unpivot das colunas de presença por dia → granularidade aluno × dia
6. **Regras de negócio:** status_inscricao, resultado_final, flags KPI
7. Salva tabela Delta `gold_curso_motorista` no Fabric

In [18]:
%run ./nb_utils_api_acto_gestao

StatementMeta(, 9ae62fad-efd6-41c8-b3b7-e0d9d0e8c496, 26, Finished, Available, Finished, True)

In [19]:
# ---------------------------------------------------------------------------
# 0. Imports, utilitários e caminhos
# ---------------------------------------------------------------------------

# Funções de limpeza: tratar_nome_colunas, colunas_para_snake_case,
# consolidar_conceito_bfill, converter_colunas_data

from collections import defaultdict
import unicodedata
import pandas as pd
import re
import os
import numpy as np

# Caminhos no Fabric (Lakehouse default)
BASE_SILVER      = "/lakehouse/default/Files/silver/santos_curso_motorista/"
NOME_TABELA_GOLD = "gold_curso_motorista"

# Lê o parquet mais recente 
arquivos = sorted([
    f for f in os.listdir(BASE_SILVER)
    if f.startswith("silver_solicitacoes") and f.endswith(".parquet")
])
arquivo_mais_recente = os.path.join(BASE_SILVER, arquivos[-1])
df = pd.read_parquet(arquivo_mais_recente)

print(f"Arquivo lido : {arquivos[-1]}")
print(f"Shape        : {df.shape[0]} linhas | {df.shape[1]} colunas")


StatementMeta(, 9ae62fad-efd6-41c8-b3b7-e0d9d0e8c496, 27, Finished, Available, Finished, False)

Arquivo lido : silver_solicitacoes.parquet
Shape        : 113 linhas | 110 colunas


In [20]:
# ---------------------------------------------------------------------------
# 1. Funções de limpeza não presentes no nb_utils
# ---------------------------------------------------------------------------

def tratar_nome_colunas(df):
    """
    Remove sufixo |ID das colunas da API Acto e adiciona _d1/_d2...
    em colunas repetidas por dia (presença, etapa, instrutor, etc.).
    """
    df = df.copy()
    bases = []
    for c in df.columns:
        m = re.match(r"^([^|(\[]+)", str(c).strip())
        base = m.group(1).strip().rstrip(":") if m else str(c).strip()
        bases.append(base)

    total_counts = defaultdict(int)
    for b in bases:
        total_counts[b] += 1

    termos_diarios = [
        "presenca", "etapa", "horario_de_inicio", "horario_de_termino",
        "data_de_inicio", "data_de_termino", "instrutor", "justificativa", "nome",
    ]

    current_counts = defaultdict(int)
    renomear = {}
    for original, base in zip(df.columns, bases):
        current_counts[base] += 1
        base_norm = (
            unicodedata.normalize("NFKD", base)
            .encode("ascii", "ignore").decode().lower()
        )
        base_norm = re.sub(r"[^a-z0-9]", "_", base_norm).strip("_")
        if total_counts[base] > 1 and any(t in base_norm for t in termos_diarios):
            renomear[original] = f"{base}_d{current_counts[base]}"
        else:
            renomear[original] = base
    return df.rename(columns=renomear)

def converter_colunas_data(df, colunas=None):
    """Converte colunas de data para datetime."""
    df = df.copy()
    colunas = colunas or [c for c in df.columns if "data" in c]
    for c in colunas:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], format="mixed", dayfirst=True, errors="coerce")
    return df

def consolidar_conceito_bfill(df):
    """Para colunas com mesmo nome (duplicatas da API), consolida via bfill."""
    grupos = defaultdict(list)
    for i, col in enumerate(df.columns):
        grupos[col].append(i)
    first_idx = {}
    for i, col in enumerate(df.columns):
        if col not in first_idx:
            first_idx[col] = i
    result = {}
    for nome in sorted(first_idx, key=lambda n: first_idx[n]):
        indices = grupos[nome]
        if len(indices) == 1:
            result[nome] = df.iloc[:, indices[0]]
        else:
            sub = df.iloc[:, indices].bfill(axis=1)
            result[nome] = sub.iloc[:, 0]
    return pd.DataFrame(result, index=df.index)

print("Funcoes de limpeza definidas: tratar_nome_colunas, converter_colunas_data")

StatementMeta(, 9ae62fad-efd6-41c8-b3b7-e0d9d0e8c496, 28, Finished, Available, Finished, False)

Funcoes de limpeza definidas: tratar_nome_colunas, converter_colunas_data


In [21]:
# ---------------------------------------------------------------------------
# 2. Pipeline de limpeza
# ---------------------------------------------------------------------------

# Renomeia colunas de presença pelo ID fixo do campo antes de tratar_nome_colunas.
# Garante mapeamento correto (presenca_d1..d8) independente da ordem da API.
# Fonte dos IDs: payload_santos_curso_motorista.json — campo "sequencia" de cada CBO_PRESENCA
RENAME_PRESENCA = {
    "Presença:|60":  "presenca_d1",   # GESTÃO 1º DIA
    "Presença:|124": "presenca_d2",   # GESTÃO 2º DIA
    "Presença:|187": "presenca_d3",   # GESTÃO 3º DIA
    "Presença:|250": "presenca_d4",   # GESTÃO 4º DIA
    "Presença:|313": "presenca_d5",   # GESTÃO 5º DIA
    "Presença:|506": "presenca_d6",   # GESTÃO 6º DIA
    "Presença:|569": "presenca_d7",   # GESTÃO 7º DIA
    "Presença:|596": "presenca_d8",   # AVALIAÇÃO
}
df = df.rename(columns={k: v for k, v in RENAME_PRESENCA.items() if k in df.columns})

# Renomeia colunas de NOME DOS ALUNOS por dia (antes de ajustar_nome_colunas)
# Essas colunas contêm o nome do aluno participante em cada dia da gestão
# Fonte dos IDs: payload_santos_curso_motorista.json — campo "sequencia" de cada TXT_NOME_INTERESSADO
RENAME_NOME = {
    "Nome:|54":   "nome_d1",          # GESTÃO 1º DIA (vazio — dia administrativo)
    "Nome:|118":  "nome_d2",          # GESTÃO 2º DIA (52 alunos)
    "Nome:|181":  "nome_d3",          # GESTÃO 3º DIA (51 alunos)
    "Nome:|244":  "nome_d4",          # GESTÃO 4º DIA (51 alunos)
    "Nome:|307":  "nome_d5",          # GESTÃO 5º DIA (51 alunos)
    "Nome:|500":  "nome_d6",          # GESTÃO 6º DIA (51 alunos)
    "Nome:|563":  "nome_d7",          # GESTÃO 7º DIA (51 alunos)
    "Nome:|594":  "nome_d8",          # AVALIAÇÃO (74 alunos)
}
df = df.rename(columns={k: v for k, v in RENAME_NOME.items() if k in df.columns})

df = tratar_nome_colunas(df)       # strip |ID, adiciona _d1/_d2 em colunas repetidas
df = ajustar_nome_colunas(df)      # snake_case sem acentos (utils)
df = consolidar_conceito_bfill(df)  # consolida colunas duplicadas da API 
df = converter_colunas_data(df, [
    c for c in ["data_finalizacao", "data_criacao", "data_inicio", "data_fim"]
    if c in df.columns
])

print(f"Shape apos limpeza: {df.shape[0]} linhas | {df.shape[1]} colunas")

StatementMeta(, 9ae62fad-efd6-41c8-b3b7-e0d9d0e8c496, 29, Finished, Available, Finished, False)

Shape apos limpeza: 113 linhas | 106 colunas


In [22]:
# ---------------------------------------------------------------------------
# 3. Inspeção — identifica colunas de presença e campos-chave
#    Execute esta célula e use o resultado para confirmar os COL_* abaixo
# ---------------------------------------------------------------------------

cols_presenca = sorted([c for c in df.columns if "presenca" in c])
cols_data     = sorted([c for c in df.columns if "data" in c])

print(f"Colunas de presenca detectadas ({len(cols_presenca)}):")
for c in cols_presenca:
    print(f"  {c}  ->  valores unicos: {df[c].dropna().unique()[:8]}")

print(f"\nColunas de data : {cols_data}")
print(f"\nTodas as colunas:")
for c in df.columns:
    print(f"  {c}")
print(f"\nAmostra (3 linhas):")
display(df.head(3))

StatementMeta(, 9ae62fad-efd6-41c8-b3b7-e0d9d0e8c496, 30, Finished, Available, Finished, False)

Colunas de presenca detectadas (8):
  presenca_d1  ->  valores unicos: ['']
  presenca_d2  ->  valores unicos: ['Presente' '']
  presenca_d3  ->  valores unicos: ['Presente' '']
  presenca_d4  ->  valores unicos: ['Presente' '']
  presenca_d5  ->  valores unicos: ['Presente' '']
  presenca_d6  ->  valores unicos: ['Presente' '']
  presenca_d7  ->  valores unicos: ['Presente' '']
  presenca_d8  ->  valores unicos: ['' 'Presente' 'Ausente']

Colunas de data : ['data_criacao', 'data_de_início_d1', 'data_de_início_d2', 'data_de_início_d3', 'data_de_início_d4', 'data_de_início_d5', 'data_de_início_d6', 'data_de_início_d7', 'data_de_início_d8', 'data_de_término_d1', 'data_de_término_d2', 'data_de_término_d3', 'data_de_término_d4', 'data_de_término_d5', 'data_de_término_d6', 'data_de_término_d7', 'data_de_término_d8', 'data_finalizacao']

Todas as colunas:
  n_solicitacao
  servico
  status_fluxo
  data_finalizacao
  data_criacao
  solicitante
  etapa_d1
  data_de_início_d1
  data_de_término_

SynapseWidget(Synapse.DataFrame, dc8b4a3d-2b80-4395-b12f-4530a429a417)

In [23]:
# ---------------------------------------------------------------------------
# 3.1 Consolidação de NOME_ALUNO (novo campo)
#      Consolida nomes de d2..d8 (dias de aula com alunos presentes)
#      Mantém solicitante para auditoria, mas nome_aluno é a fonte truth
# ---------------------------------------------------------------------------

# Colunas de nome por dia (após ajustar_nome_colunas, estão em snake_case)
cols_nome_dias = [f"nome_d{i}" for i in range(2, 9)]  # d2..d8 (d1 é administrativo vazio)
cols_nome_dias = [c for c in cols_nome_dias if c in df.columns]

print(f"Colunas de nome encontradas: {cols_nome_dias}")

# Consolida nome_aluno: busca nome não-vazio em d2..d8, ordem de prioridade d2→d8
def consolidar_nome_aluno(row):
    for col in cols_nome_dias:
        valor = row.get(col) if isinstance(row, dict) else (row[col] if hasattr(row, '__getitem__') else getattr(row, col, None))
        if valor and isinstance(valor, str) and valor.strip():
            return valor.strip()
    # Fallback: usa solicitante se nenhum nome diário encontrado
    fallback = row.get('solicitante') if isinstance(row, dict) else (row['solicitante'] if 'solicitante' in row else None)
    return fallback.strip() if fallback and isinstance(fallback, str) else "(sem nome)"

df["nome_aluno"] = df[cols_nome_dias + ["solicitante"]].apply(consolidar_nome_aluno, axis=1)

print(f"\nConsolidacao de nome_aluno:")
print(f"  Valores unicos encontrados: {df['nome_aluno'].nunique()}")
print(f"  Amostra de valores:")
print(df["nome_aluno"].value_counts().head(10))


StatementMeta(, 9ae62fad-efd6-41c8-b3b7-e0d9d0e8c496, 31, Finished, Available, Finished, False)

Colunas de nome encontradas: ['nome_d2', 'nome_d3', 'nome_d4', 'nome_d5', 'nome_d6', 'nome_d7', 'nome_d8']

Consolidacao de nome_aluno:
  Valores unicos encontrados: 93
  Amostra de valores:
nome_aluno
GILBERTO BRITTO ALBERTI        4
MÁRCIO RODRIGUES ALVES         3
SERGIO MAGALHAES MENDES        3
PAULO SÉRGIO LOPES             2
Tomaz Vitor da Silva Aranha    2
SIDINEI LIMA SANTOS            2
PAULO DE JESUS REIS            2
Ana Paula Lima De Freitas      2
LUIZ HENRIQUE DA C TENORIO     2
José Oliveira fontes           2
Name: count, dtype: int64


In [24]:
# ---------------------------------------------------------------------------
# 4. Unpivot das colunas de presença por dia
#    Dias de aula: presenca_d2..d8 (d1 é dia administrativo sem presença real)
#    Nomenclatura no dashboard: D1..D7  (presenca_d{N} → D{N-1})
#    Resultado: 1 linha por aluno × dia de aula real
# ---------------------------------------------------------------------------

# Somente dias de aula reais (exclui presenca_d1 — dia administrativo)
COLS_PRESENCA_DIA = [f"presenca_d{i}" for i in range(2, 9)]  # d2..d8
cols_presenca_dia = [c for c in COLS_PRESENCA_DIA if c in df.columns]
print(f"Colunas de presenca para unpivot: {cols_presenca_dia}")

# Mapeamento observações por dia (D1=obs do 2º dia, ..., D7=obs do 8º dia)
MAPA_OBS_DIA = {
    "D1": "observacoes_2o_dia", "D2": "observacoes_3o_dia",
    "D3": "observacoes_4o_dia", "D4": "observacoes_5o_dia",
    "D5": "observacoes_6o_dia", "D6": "observacoes_7o_dia",
    "D7": "observacoes_8o_dia",
}

cols_obs = [c for c in MAPA_OBS_DIA.values() if c in df.columns]
id_vars  = [c for c in df.columns if c not in cols_presenca_dia and c not in cols_obs]

if cols_presenca_dia:
    df_long = df.melt(
        id_vars=id_vars,
        value_vars=cols_presenca_dia,
        var_name="dia_col",
        value_name="presenca_valor",
    )

    # presenca_d{N} -> D{N-1}: d2->D1, d3->D2, ..., d8->D7
    df_long["dia_curso"] = "D" + (
        df_long["dia_col"].str.extract(r"(\d+)", expand=False).astype(int) - 1
    ).astype(str)
    df_long = df_long.drop(columns=["dia_col"])

    # observacao_dia por merge vetorizado (evita apply linha a linha — lento em datasets grandes)
    if cols_obs:
        df_obs  = df[[COL_ID] + cols_obs].drop_duplicates(COL_ID)
        df_long = df_long.merge(df_obs, on=COL_ID, how="left")

        df_long["observacao_dia"] = ""
        for dia, col_obs in MAPA_OBS_DIA.items():
            if col_obs in df_long.columns:
                mask = df_long["dia_curso"] == dia
                df_long.loc[mask, "observacao_dia"] = (
                    df_long.loc[mask, col_obs].fillna("").astype(str)
                    .where(lambda s: ~s.str.lower().isin(["", "nan", "none"]), "")
                )
        df_long = df_long.drop(columns=cols_obs)
    else:
        df_long["observacao_dia"] = ""

    print(f"Shape apos unpivot: {df_long.shape[0]} linhas | {df_long.shape[1]} colunas")
    print(f"Dias gerados      : {sorted(df_long['dia_curso'].unique())}")
    display(df_long.head(8))
else:
    df_long = df.copy()
    df_long["dia_curso"]      = "Sem Dados"
    df_long["presenca_valor"] = ""
    df_long["observacao_dia"] = ""
    print("ATENCAO: colunas presenca_dN nao encontradas.")

StatementMeta(, 9ae62fad-efd6-41c8-b3b7-e0d9d0e8c496, 32, Finished, Available, Finished, False)

Colunas de presenca para unpivot: ['presenca_d2', 'presenca_d3', 'presenca_d4', 'presenca_d5', 'presenca_d6', 'presenca_d7', 'presenca_d8']
Shape apos unpivot: 791 linhas | 103 colunas
Dias gerados      : ['D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7']


SynapseWidget(Synapse.DataFrame, 2bb1d5a1-c543-446c-88cf-c7acddcc1fd6)

In [25]:
# ---------------------------------------------------------------------------
# 5. CONFIGURAÇÃO DAS REGRAS DE NEGÓCIO
#    Campos mapeados diretamente dos dados reais inspecionados
# ---------------------------------------------------------------------------

# --- Identificação ---
COL_ID           = "n_solicitacao"       # chave única da inscrição
COL_SOLICITANTE  = "solicitante"          # [AUDITORIA] nome de quem abriu a OS administrativamente
COL_NOME_ALUNO   = "nome_aluno"           # [ANALISE] nome consolidado do aluno/participante (verdade)
COL_SERVICO      = "servico"              # código do serviço
COL_NOME_SERV    = "nome_servico"         # nome do serviço
COL_TIPO_CURSO   = "tipo_de_curso"        # tipo do curso (usado como TURMA)
COL_STATUS_FLUXO = "status_fluxo"         # status da OS — NÃO é status do aluno
COL_CONCLUSAO    = "conclusao"            # CBO_CONCLUSAO_CURSO: "Aprovado" | "Reprovado"
COL_MOTIVO_REP   = "motivo_da_reprovacao" # motivo de reprovação
COL_CARGA_HOR    = "carga_horaria"        # carga horária
COL_DATA_INSC    = "data_criacao"         # data de criação/inscrição
COL_DATA_FIM     = "data_finalizacao"     # data de finalização

# --- Coluna de etapa RECUSADOS ---
# Candidatos recusados na seleção têm a etapa RECUSADOS finalizada.
# Após tratar_nome_colunas, Etapa|616 → etapa_d11.
# ATENÇÃO: verificar se o sufixo _d11 se mantém após re-ingestão no Fabric.
COL_ETAPA_RECUSADOS = "etapa_d11"

# --- Colunas de avaliação (Resultados das Avaliações) ---
COL_AVALIACAO_NOTA    = "avaliacao_do_curso"
COL_CARGA_HOR_OPINIAO = "a_carga_horaria_do_curso_foi"
COL_INSTRUTORES       = "voce_sentiu_que_os_instrutores_demonstraram_interesse_pelo_progresso_dos_participantes?"
COL_CURSO_CANSATIVO   = "voce_sentiu_em_algum_momento_que_o_curso_se_tornou_cansativo?"
COL_OPINIAO_CURSO     = "na_sua_opiniao_o_curso_foi"
COL_MATERIAS          = "as_materias_e_as_instrucoes_foram_dadas_de_forma"
COL_COMO_SOUBE        = "como_voce_ficou_sabendo_sobre_o_curso"
COL_COMENTARIOS       = "espaco_reservado_para_comentarios_críticas_e_sugestões"

COLS_AVALIACAO = [
    COL_AVALIACAO_NOTA, COL_CARGA_HOR_OPINIAO, COL_INSTRUTORES,
    COL_CURSO_CANSATIVO, COL_OPINIAO_CURSO, COL_MATERIAS,
    COL_COMO_SOUBE, COL_COMENTARIOS,
]

# --- Presença ---
# presenca_d1 : dia administrativo — sem presença real, excluído do dashboard
# presenca_d2..d8 : dias de aula reais → exibidos como D1..D7 no dashboard
TODOS_DIAS_PRESENCA = [f"presenca_d{i}" for i in range(1, 9)]
DIAS_AULA_VALIDOS   = [f"presenca_d{i}" for i in range(2, 9)]

VALORES_PRESENTE     = ["Presente", "presente", "PRESENTE", "Sim", "sim", "S", "P", "1"]
STATUS_CANCELADO_OS  = ["Cancelado", "cancelado", "CANCELADO", "Rascunho", "rascunho"]
STATUS_FINALIZADO_OS = ["Finalizado", "finalizado", "FINALIZADO"]

print("Configuracao definida.")
print(f"  Turma (campo)           : {COL_TIPO_CURSO}")
print(f"  Nome do aluno (campo)   : {COL_NOME_ALUNO}  <- fonte TRUTH de quem estudou")
print(f"  Conclusao (campo)       : {COL_CONCLUSAO}  <- fonte UNICA de Aprovado/Reprovado")
print(f"  Etapa Recusados (campo) : {COL_ETAPA_RECUSADOS}")
print(f"  Dias de aula validos    : {DIAS_AULA_VALIDOS}")
print(f"  Exibidos no dashboard   : D1..D7  (presenca_d2 -> D1, ..., presenca_d8 -> D7)")

StatementMeta(, 9ae62fad-efd6-41c8-b3b7-e0d9d0e8c496, 33, Finished, Available, Finished, False)

Configuracao definida.
  Turma (campo)           : tipo_de_curso
  Nome do aluno (campo)   : nome_aluno  <- fonte TRUTH de quem estudou
  Conclusao (campo)       : conclusao  <- fonte UNICA de Aprovado/Reprovado
  Etapa Recusados (campo) : etapa_d11
  Dias de aula validos    : ['presenca_d2', 'presenca_d3', 'presenca_d4', 'presenca_d5', 'presenca_d6', 'presenca_d7', 'presenca_d8']
  Exibidos no dashboard   : D1..D7  (presenca_d2 -> D1, ..., presenca_d8 -> D7)


In [26]:
# ---------------------------------------------------------------------------
# 6. Regra de Deferimento
#
# Prioridade: Indeferido > Cancelado > Deferido
#
# Indeferido  = candidato recusado na seleção (etapa RECUSADOS finalizada)
#               Nunca frequentou o curso.
# Cancelado   = OS encerrada administrativamente antes de iniciar
# Deferido    = inscrição aceita — aluno ingressou no curso
# ---------------------------------------------------------------------------

# Detecta candidatos indeferidos pela etapa RECUSADOS
ids_indeferidos = set()
if COL_ETAPA_RECUSADOS in df_long.columns:
    ids_indeferidos = set(
        df_long.loc[
            df_long[COL_ETAPA_RECUSADOS].fillna("").str.strip().str.upper() == "RECUSADOS",
            COL_ID,
        ]
    )
print(f"Indeferidos detectados (etapa RECUSADOS): {len(ids_indeferidos)}")
print(f"Valores unicos em '{COL_STATUS_FLUXO}':")
print(df_long[COL_STATUS_FLUXO].value_counts(dropna=False))
print()

def calcular_deferimento(row):
    if row[COL_ID] in ids_indeferidos:
        return "Indeferido"
    v = row[COL_STATUS_FLUXO]
    if pd.isna(v):
        return "Cancelado"
    return "Cancelado" if str(v).strip() in STATUS_CANCELADO_OS else "Deferido"

df_long["status_deferimento"] = df_long.apply(calcular_deferimento, axis=1)

print("status_deferimento (alunos unicos):")
print(df_long.groupby("status_deferimento")[COL_ID].nunique())

StatementMeta(, 9ae62fad-efd6-41c8-b3b7-e0d9d0e8c496, 34, Finished, Available, Finished, False)

Indeferidos detectados (etapa RECUSADOS): 17
Valores unicos em 'status_fluxo':
status_fluxo
Finalizado              665
Cancelado                84
Pendente atendimento     35
Em atendimento            7
Name: count, dtype: int64

status_deferimento (alunos unicos):
status_deferimento
Cancelado     12
Deferido      84
Indeferido    17
Name: n_solicitacao, dtype: int64


In [27]:
# ---------------------------------------------------------------------------
# 7. Padronização da presença por dia
#
# Cancelados, Indeferidos e Em Atendimento → status_presenca = "N/A"
#   - Cancelado/Indeferido : nunca participaram do curso
#   - Em Atendimento       : curso em andamento — presença parcial
#                            não deve ser contada como ausência definitiva
#
# flag_presente / flag_ausente usados nos gráficos de frequência.
# IMPORTANTE: presença serve apenas para FISCALIZAÇÃO.
#             Não determina aprovação/reprovação — isso vem de COL_CONCLUSAO.
# ---------------------------------------------------------------------------

presenca_valor     = df_long["presenca_valor"].fillna("").astype(str).str.strip()
status_deferimento = df_long["status_deferimento"].fillna("").astype(str)

mask_nao_ativo = (
    status_deferimento.isin(["Cancelado", "Indeferido"]) |
    (df_long[COL_STATUS_FLUXO].fillna("").str.lower() == "em atendimento")
)
mask_presente = presenca_valor.isin(VALORES_PRESENTE)

df_long["status_presenca"] = "Ausente"
df_long.loc[mask_presente & ~mask_nao_ativo, "status_presenca"] = "Presente"
df_long.loc[mask_nao_ativo,                  "status_presenca"] = "N/A"

df_long["flag_presente"]  = (df_long["status_presenca"] == "Presente").astype(int)
df_long["flag_ausente"]   = (df_long["status_presenca"] == "Ausente").astype(int)
df_long["flag_dia_aula"]  = df_long["dia_curso"].isin([f"D{i}" for i in range(1, 8)]).astype(int)

print("Presenca por dia — Deferidos com curso concluido:")
print(
    df_long[
        (df_long["flag_dia_aula"] == 1) &
        (df_long["status_deferimento"] == "Deferido") &
        (df_long[COL_STATUS_FLUXO].fillna("").str.lower() != "em atendimento")
    ]
    .groupby(["dia_curso", "status_presenca"])[COL_ID]
    .count().unstack(fill_value=0)
)

StatementMeta(, 9ae62fad-efd6-41c8-b3b7-e0d9d0e8c496, 35, Finished, Available, Finished, False)

Presenca por dia — Deferidos com curso concluido:
status_presenca  Ausente  Presente
dia_curso                         
D1                    33        50
D2                    33        50
D3                    33        50
D4                    33        50
D5                    33        50
D6                    33        50
D7                    42        41


In [28]:
# ---------------------------------------------------------------------------
# 8. Regra de status_inscricao
#
# Prioridade: Indeferido > Cancelado > Aprovado > Reprovado > Em Andamento > Sem Informacoes
#
# Aprovado       : OS Finalizada + conclusao positiva OU 100% presença
# Reprovado      : OS Finalizada + conclusao negativa OU presença parcial
# Em Andamento   : OS Em atendimento (independente do estado de presença)
# Indeferido     : etapa RECUSADOS finalizada — nunca frequentou
# Cancelado      : OS cancelada administrativamente
# Sem Informacoes: OS Finalizada sem conclusao e sem presença — dado não
#                  preenchido na fonte, investigar com o operacional
# ---------------------------------------------------------------------------

dias_cols = [c for c in [f"presenca_d{i}" for i in range(2, 9)] if c in df.columns]
df_aluno  = (
    df[[COL_ID, COL_STATUS_FLUXO, COL_CONCLUSAO] + dias_cols]
    .drop_duplicates(subset=[COL_ID])
    .copy()
)

# Máscaras de presença (sobre o df não-unpivotado para evitar duplicatas por dia)
presenca_norm   = df_aluno[dias_cols].fillna("").astype(str).apply(lambda col: col.str.strip())
mask_todas_pres = presenca_norm.isin(VALORES_PRESENTE).all(axis=1)
mask_sem_pres   = (presenca_norm == "").all(axis=1)

# Máscaras de status do fluxo
mask_finalizado     = df_aluno[COL_STATUS_FLUXO].isin(STATUS_FINALIZADO_OS)
mask_em_atendimento = df_aluno[COL_STATUS_FLUXO].str.lower() == "em atendimento"
mask_cancelado      = df_aluno[COL_STATUS_FLUXO].isin(STATUS_CANCELADO_OS)
mask_indeferido_al  = df_aluno[COL_ID].isin(ids_indeferidos)  # definido na célula 6

# ── Classificação por prioridade ──────────────────────────────────────────────
df_aluno["status_inscricao"] = "Sem Informacoes"

# 1. Indeferido — recusado na seleção, não frequentou o curso
df_aluno.loc[mask_indeferido_al, "status_inscricao"] = "Indeferido"

# 2. Cancelado — cancelamento administrativo da OS
df_aluno.loc[mask_cancelado, "status_inscricao"] = "Cancelado"

# Normaliza campo conclusão (case-insensitive, sem espaços)
conclusao_norm = df_aluno[COL_CONCLUSAO].fillna("").astype(str).str.strip().str.lower()

base = ~mask_cancelado & ~mask_indeferido_al & mask_finalizado

# 3. Aprovado — OS Finalizada + conclusão positiva OU todas as presenças marcadas
mask_aprovado = base & (
    conclusao_norm.isin(["aprovado", "aprovada", "concluido", "concluído", "finalizado", "concluinte"])
    | mask_todas_pres
)

# 4. Reprovado — OS Finalizada + conclusão negativa OU presença parcial
mask_reprovado = base & (
    conclusao_norm.isin(["reprovado", "reprovada", "nao aprovado", "não aprovado"])
    | (~mask_todas_pres & ~mask_sem_pres)
)

# 5. Em Andamento — OS em atendimento (curso ainda em curso)
mask_em_and = ~mask_cancelado & ~mask_indeferido_al & mask_em_atendimento

df_aluno.loc[mask_aprovado,  "status_inscricao"] = "Aprovado"
df_aluno.loc[mask_reprovado, "status_inscricao"] = "Reprovado"
df_aluno.loc[mask_em_and,    "status_inscricao"] = "Em Andamento"

# Diagnóstico de "Sem Informações" — OS Finalizada sem dados preenchidos
n_sem = (df_aluno["status_inscricao"] == "Sem Informacoes").sum()
if n_sem > 0:
    ids_sem = df_aluno.loc[df_aluno["status_inscricao"] == "Sem Informacoes", COL_ID].tolist()
    print(f"ATENCAO: {n_sem} aluno(s) com 'Sem Informacoes'.")
    print(f"  Causa: OS Finalizada sem presenca/conclusao preenchida na fonte.")
    print(f"  IDs: {ids_sem}")

# Merge de volta para df_long
df_long = df_long.merge(df_aluno[[COL_ID, "status_inscricao"]], on=COL_ID, how="left")

print("\nstatus_inscricao (alunos unicos):")
print(df_long.groupby("status_inscricao")[COL_ID].nunique())

StatementMeta(, 9ae62fad-efd6-41c8-b3b7-e0d9d0e8c496, 36, Finished, Available, Finished, False)

ATENCAO: 16 aluno(s) com 'Sem Informacoes'.
  Causa: OS Finalizada sem presenca/conclusao preenchida na fonte.
  IDs: ['884235', '884469', '885056', '885593', '885955', '887354', '887419', '887942', '888393', '888434', '888963', '932768', '933057', '933153', '933315', '933939']

status_inscricao (alunos unicos):
status_inscricao
Aprovado           41
Cancelado          12
Em Andamento        1
Indeferido         17
Reprovado          26
Sem Informacoes    16
Name: n_solicitacao, dtype: int64


In [32]:
# ---------------------------------------------------------------------------
# 9. Flags de KPI, turma, avaliações e calendário
# ---------------------------------------------------------------------------

# ── Flags binárias ────────────────────────────────────────────────────────────
df_long["flag_deferido"]     = (df_long["status_deferimento"] == "Deferido").astype(int)
df_long["flag_indeferido"]   = (df_long["status_deferimento"] == "Indeferido").astype(int)
df_long["flag_cancelado"]    = (df_long["status_deferimento"] == "Cancelado").astype(int)
df_long["flag_aprovado"]     = (df_long["status_inscricao"]  == "Aprovado").astype(int)
df_long["flag_reprovado"]    = (df_long["status_inscricao"]  == "Reprovado").astype(int)
df_long["flag_em_andamento"] = (df_long["status_inscricao"]  == "Em Andamento").astype(int)
df_long["flag_sem_info"]     = (df_long["status_inscricao"]  == "Sem Informacoes").astype(int)
# Concluinte = chegou ao fim com decisão registrada (Aprovado ou Reprovado)
df_long["flag_concluinte"]   = (
    (df_long["flag_aprovado"] == 1) | (df_long["flag_reprovado"] == 1)
).astype(int)

# ── Turma ─────────────────────────────────────────────────────────────────────
if COL_TIPO_CURSO in df_long.columns:
    df_long["turma"] = (
        df_long[COL_TIPO_CURSO].str.strip().replace("", "Sem Turma").fillna("Sem Turma")
    )
else:
    df_long["turma"] = "Sem Turma"

# ── Avaliação numérica ────────────────────────────────────────────────────────
if COL_AVALIACAO_NOTA in df_long.columns:
    df_long[COL_AVALIACAO_NOTA] = pd.to_numeric(df_long[COL_AVALIACAO_NOTA], errors="coerce")

# ── Datas de inscrição ────────────────────────────────────────────────────────
if COL_DATA_INSC in df_long.columns:
    df_long["ano_inscricao"] = df_long[COL_DATA_INSC].dt.year
    df_long["mes_inscricao"] = df_long[COL_DATA_INSC].dt.month
    df_long["mes_nome"]      = df_long[COL_DATA_INSC].dt.strftime("%B")

# ── KPIs de verificação ───────────────────────────────────────────────────────
total   = df_long[COL_ID].nunique()
defer_  = df_long.loc[df_long["flag_deferido"]     == 1, COL_ID].nunique()
indefer = df_long.loc[df_long["flag_indeferido"]   == 1, COL_ID].nunique()
cancel  = df_long.loc[df_long["flag_cancelado"]    == 1, COL_ID].nunique()
aprov   = df_long.loc[df_long["flag_aprovado"]     == 1, COL_ID].nunique()
reprov  = df_long.loc[df_long["flag_reprovado"]    == 1, COL_ID].nunique()
em_and  = df_long.loc[df_long["flag_em_andamento"] == 1, COL_ID].nunique()
s_info  = df_long.loc[df_long["flag_sem_info"]     == 1, COL_ID].nunique()
concl   = df_long.loc[df_long["flag_concluinte"]   == 1, COL_ID].nunique()

tx_def = defer_  / total  if total  > 0 else 0
# Evasão = deferidos sem resultado conclusivo / total deferidos
tx_eva = (defer_ - concl) / defer_ if defer_ > 0 else 0

print("--- KPIs ---")
print(f"  Total Inscricoes  : {total}")
print(f"  Deferidos         : {defer_}  ({tx_def:.1%})")
print(f"  Indeferidos       : {indefer}")
print(f"  Cancelados        : {cancel}")
print(f"  Aprovados         : {aprov}")
print(f"  Reprovados        : {reprov}")
print(f"  Em Andamento      : {em_and}")
print(f"  Sem Informacoes   : {s_info}")
print(f"  Taxa de Evasao    : {tx_eva:.2%}  (deferidos sem resultado / deferidos)")
print()

soma_def = aprov + reprov + em_and + s_info
soma_tot = defer_ + indefer + cancel
print(f"  Deferidos ({defer_}) == Aprov+Reprov ({aprov+reprov}) + Em And ({em_and}) + Sem Info ({s_info}) = {soma_def}: {'OK' if defer_ == soma_def else 'DIVERGENCIA'}")
print(f"  Total ({total}) == Deferidos ({defer_}) + Indeferidos ({indefer}) + Cancelados ({cancel}) = {soma_tot}: {'OK' if total == soma_tot else 'DIVERGENCIA'}")

# ---------------------------------------------------------------------------
# df_gold = snapshot final de df_long apos todas as transformacoes
# ---------------------------------------------------------------------------
df_gold = df_long.copy()
print(f"df_gold criado: {df_gold.shape[0]:,} linhas | {df_gold.shape[1]} colunas")

StatementMeta(, 9ae62fad-efd6-41c8-b3b7-e0d9d0e8c496, 40, Finished, Available, Finished, False)

--- KPIs ---
  Total Inscricoes  : 113
  Deferidos         : 84  (74.3%)
  Indeferidos       : 17
  Cancelados        : 12
  Aprovados         : 41
  Reprovados        : 26
  Em Andamento      : 1
  Sem Informacoes   : 16
  Taxa de Evasao    : 20.24%  (deferidos sem resultado / deferidos)

  Deferidos (84) == Aprov+Reprov (67) + Em And (1) + Sem Info (16) = 84: OK
  Total (113) == Deferidos (84) + Indeferidos (17) + Cancelados (12) = 113: OK
df_gold criado: 791 linhas | 121 colunas


In [33]:
print()
nulos = df_gold[[COL_ID, COL_NOME_ALUNO, "turma", "status_deferimento",
                 "status_inscricao", "dia_curso", "status_presenca",
                 "ano_inscricao", "mes_inscricao"]].isnull().sum()
print("Nulos nos campos essenciais:")
print(nulos[nulos > 0] if nulos.any() else "  Sem nulos.")
print()
display(df_gold[[COL_ID, COL_NOME_ALUNO, "turma", "status_deferimento",
                 "status_inscricao", "dia_curso", "status_presenca",
                 "flag_presente", "flag_ausente",
                 "ano_inscricao", "mes_nome"]].head(10))

StatementMeta(, 9ae62fad-efd6-41c8-b3b7-e0d9d0e8c496, 41, Finished, Available, Finished, False)


Nulos nos campos essenciais:
  Sem nulos.



SynapseWidget(Synapse.DataFrame, 11f19aef-9745-4554-97f0-312b58d309d2)

In [34]:
# ---------------------------------------------------------------------------
# 11. Salva tabela Gold no Lakehouse Fabric (Delta)
#     Converte Pandas -> Spark, depois salva como Delta Table
# ---------------------------------------------------------------------------

# Seleciona apenas as colunas relevantes para o painel
COLS_GOLD = [
    # Identificacao
    COL_ID, COL_NOME_ALUNO, COL_SOLICITANTE,
    # Calendario e presenca
    "turma", "dia_curso", "status_presenca",
    # Regras de negocio
    "status_deferimento", "status_inscricao",
    # Filtros de data (numericos + nome do mes)
    "ano_inscricao", "mes_inscricao", "mes_nome",
    # Datas brutas (Time Intelligence no Power BI)
    COL_DATA_INSC, COL_DATA_FIM,
    # Flags binarias
    "flag_deferido", "flag_indeferido", "flag_cancelado",
    "flag_aprovado", "flag_reprovado", "flag_em_andamento",
    "flag_sem_info", "flag_concluinte",
    "flag_presente", "flag_ausente", "flag_dia_aula",
    # Avaliacao numerica
    COL_AVALIACAO_NOTA,
    # Pesquisa de satisfacao (usadas nas medidas DAX)
    COL_CARGA_HOR_OPINIAO, COL_INSTRUTORES, COL_CURSO_CANSATIVO,
    COL_OPINIAO_CURSO, COL_MATERIAS, COL_COMO_SOUBE, COL_COMENTARIOS,
]

# Filtra apenas colunas que existem (seguro contra re-ingestao)
COLS_GOLD = [c for c in COLS_GOLD if c in df_gold.columns]

df_gold_final = df_gold[COLS_GOLD].copy()

print(f"Shape final  : {df_gold_final.shape[0]:,} linhas | {df_gold_final.shape[1]} colunas")
print(f"Alunos unicos: {df_gold_final[COL_ID].nunique():,}")
print()

# Normaliza nomes de coluna para Spark (remove ?, acentos e chars especiais)
import unicodedata, re

def spark_safe(name):
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r"[^a-zA-Z0-9_]", "", name)
    return name.strip("_")

rename_map = {c: spark_safe(c) for c in df_gold_final.columns if c != spark_safe(c)}
if rename_map:
    print("Colunas renomeadas para compatibilidade Spark:")
    for old, new in rename_map.items():
        print(f"  {old!r} -> {new!r}")
    df_gold_final = df_gold_final.rename(columns=rename_map)
    print()

# Converte Pandas DataFrame -> Spark DataFrame
df_spark = spark.createDataFrame(df_gold_final)

# Salva na tabela Delta 'gold_curso_motorista' (Lakehouse)
try:
    (
        df_spark.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(NOME_TABELA_GOLD)
    )

    # Valida - conta linhas na tabela salva
    n_salvo = spark.table(NOME_TABELA_GOLD).count()
    print(f"Tabela '{NOME_TABELA_GOLD}' salva com sucesso!")
    print(f"   Linhas salvas: {n_salvo:,}")
    print(f"   Colunas      : {len(df_gold_final.columns)}")
    if n_salvo == len(df_gold_final):
        print(f"   Status       : Validado (linhas = esperado)")
    else:
        print(f"   Status       : ATENCAO Divergencia ({n_salvo} != {len(df_gold_final)})")

except Exception as e:
    print(f"ERRO ao salvar: {e}")
    import traceback
    traceback.print_exc()


StatementMeta(, 9ae62fad-efd6-41c8-b3b7-e0d9d0e8c496, 42, Finished, Available, Finished, False)

Shape final  : 791 linhas | 31 colunas
Alunos unicos: 113

Colunas renomeadas para compatibilidade Spark:
  'voce_sentiu_que_os_instrutores_demonstraram_interesse_pelo_progresso_dos_participantes?' -> 'voce_sentiu_que_os_instrutores_demonstraram_interesse_pelo_progresso_dos_participantes'
  'voce_sentiu_em_algum_momento_que_o_curso_se_tornou_cansativo?' -> 'voce_sentiu_em_algum_momento_que_o_curso_se_tornou_cansativo'
  'espaco_reservado_para_comentarios_críticas_e_sugestões' -> 'espaco_reservado_para_comentarios_criticas_e_sugestoes'

Tabela 'gold_curso_motorista' salva com sucesso!
   Linhas salvas: 791
   Colunas      : 31
   Status       : Validado (linhas = esperado)


In [35]:
#10.1 - Verificação
print()
nulos = df_gold[[COL_ID, COL_NOME_ALUNO, "turma", "status_deferimento",
                 "status_inscricao", "dia_curso", "status_presenca",
                 "ano_inscricao", "mes_inscricao"]].isnull().sum()
print("Nulos nos campos essenciais:")
print(nulos[nulos > 0] if nulos.any() else "  Sem nulos.")
print()
display(df_gold[[COL_ID, COL_NOME_ALUNO, "turma", "status_deferimento",
                 "status_inscricao", "dia_curso", "status_presenca",
                 "flag_presente", "flag_ausente",
                 "ano_inscricao", "mes_nome"]].head(10))

StatementMeta(, 9ae62fad-efd6-41c8-b3b7-e0d9d0e8c496, 43, Finished, Available, Finished, False)


Nulos nos campos essenciais:
  Sem nulos.



SynapseWidget(Synapse.DataFrame, 8e884481-bfb6-4b8a-bf0b-859736c2bd2c)

In [36]:
# ---------------------------------------------------------------------------
# 11. Salva tabela Gold no Lakehouse Fabric (Delta)
#     Converte Pandas -> Spark, depois salva como Delta Table
# ---------------------------------------------------------------------------

# Seleciona apenas as colunas relevantes para o painel
COLS_GOLD = [
    # Identificacao
    COL_ID, COL_NOME_ALUNO, COL_SOLICITANTE,
    # Calendario e presenca
    "turma", "dia_curso", "status_presenca",
    # Regras de negocio
    "status_deferimento", "status_inscricao",
    # Filtros de data (numericos + nome do mes)
    "ano_inscricao", "mes_inscricao", "mes_nome",
    # Datas brutas (Time Intelligence no Power BI)
    COL_DATA_INSC, COL_DATA_FIM,
    # Flags binarias
    "flag_deferido", "flag_indeferido", "flag_cancelado",
    "flag_aprovado", "flag_reprovado", "flag_em_andamento",
    "flag_sem_info", "flag_concluinte",
    "flag_presente", "flag_ausente", "flag_dia_aula",
    # Avaliacao numerica
    COL_AVALIACAO_NOTA,
    # Pesquisa de satisfacao (usadas nas medidas DAX)
    COL_CARGA_HOR_OPINIAO, COL_INSTRUTORES, COL_CURSO_CANSATIVO,
    COL_OPINIAO_CURSO, COL_MATERIAS, COL_COMO_SOUBE, COL_COMENTARIOS,
]

# Filtra apenas colunas que existem (seguro contra re-ingestao)
COLS_GOLD = [c for c in COLS_GOLD if c in df_gold.columns]

df_gold_final = df_gold[COLS_GOLD].copy()

print(f"Shape final  : {df_gold_final.shape[0]:,} linhas | {df_gold_final.shape[1]} colunas")
print(f"Alunos unicos: {df_gold_final[COL_ID].nunique():,}")
print()

# Converte Pandas DataFrame -> Spark DataFrame
df_spark = spark.createDataFrame(df_gold_final)

# Salva na tabela Delta 'gold_curso_motorista' (Lakehouse)
try:
    (
        df_spark.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(NOME_TABELA_GOLD)
    )

    # Valida - conta linhas na tabela salva
    n_salvo = spark.table(NOME_TABELA_GOLD).count()
    print(f"Tabela '{NOME_TABELA_GOLD}' salva com sucesso!")
    print(f"   Linhas salvas: {n_salvo:,}")
    print(f"   Colunas      : {len(COLS_GOLD)}")
    if n_salvo == len(df_gold_final):
        print(f"   Status       : Validado (linhas = esperado)")
    else:
        print(f"   Status       : ATENCAO Divergencia ({n_salvo} != {len(df_gold_final)})")

except Exception as e:
    print(f"ERRO ao salvar: {e}")
    import traceback
    traceback.print_exc()


StatementMeta(, 9ae62fad-efd6-41c8-b3b7-e0d9d0e8c496, 44, Finished, Available, Finished, False)

Shape final  : 791 linhas | 31 colunas
Alunos unicos: 113

Tabela 'gold_curso_motorista' salva com sucesso!
   Linhas salvas: 791
   Colunas      : 31
   Status       : Validado (linhas = esperado)


In [41]:
df_gold_final.columns.tolist()

StatementMeta(, 9ae62fad-efd6-41c8-b3b7-e0d9d0e8c496, 49, Finished, Available, Finished, False)

['n_solicitacao',
 'nome_aluno',
 'solicitante',
 'turma',
 'dia_curso',
 'status_presenca',
 'status_deferimento',
 'status_inscricao',
 'ano_inscricao',
 'mes_inscricao',
 'mes_nome',
 'data_criacao',
 'data_finalizacao',
 'flag_deferido',
 'flag_indeferido',
 'flag_cancelado',
 'flag_aprovado',
 'flag_reprovado',
 'flag_em_andamento',
 'flag_sem_info',
 'flag_concluinte',
 'flag_presente',
 'flag_ausente',
 'flag_dia_aula',
 'avaliacao_do_curso',
 'a_carga_horaria_do_curso_foi',
 'voce_sentiu_que_os_instrutores_demonstraram_interesse_pelo_progresso_dos_participantes?',
 'voce_sentiu_em_algum_momento_que_o_curso_se_tornou_cansativo?',
 'na_sua_opiniao_o_curso_foi',
 'como_voce_ficou_sabendo_sobre_o_curso',
 'espaco_reservado_para_comentarios_críticas_e_sugestões']